In [1]:
from pandas.core.interchange.dataframe_protocol import DataFrame

from _bootstrap import setup

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from enum import StrEnum
import os
import pandas as pd

In [2]:
class OrientacionCorte(StrEnum):
    VERTICAL = "vertical"
    HORIZONTAL = "horizontal"

In [3]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa


# Funciones
def obtener_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin_clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"

In [4]:
def obtener_rutas_perfiles(ruta_origen: Path, clase: int | str, extension: str) -> list[Path]:
    nombre_carpeta = obtener_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [5]:
def determinar_orientacion_corte(clase: str) -> OrientacionCorte:
    if clase in range(1, 10):
        return OrientacionCorte.VERTICAL
    return OrientacionCorte.HORIZONTAL


def crear_ruta_archivo(ruta_base: Path, nombre_archivo: str, sufijo: str):
    return os.path.join(ruta_base, f"{nombre_archivo}{sufijo}.png")

In [6]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem


def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)


def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img


def obtener_porcentajes_fraccion(porcentaje_avance: float) -> list[float]:
    if porcentaje_avance <= 0 or porcentaje_avance >= 100:
        raise ValueError("porcentaje_avance debe estar entre 0 y 100 (exclusivo)")

    porcentajes: list[float] = []
    actual = porcentaje_avance
    while actual < 100:
        porcentajes.append(round(actual, 4))
        actual += porcentaje_avance

    if not porcentajes:
        raise ValueError("No se generaron fracciones con el porcentaje indicado")

    return porcentajes


def formatear_porcentaje_para_nombre(porcentaje: float) -> str:
    if float(porcentaje).is_integer():
        return str(int(porcentaje))
    return str(porcentaje).replace(".", "_")


def generar_fracciones(
        orientacion_corte: OrientacionCorte,
        porcentaje_avance: float,
) -> list[tuple[str, float]]:
    porcentajes = obtener_porcentajes_fraccion(porcentaje_avance)

    if orientacion_corte == OrientacionCorte.HORIZONTAL:
        # Horizontal: primero desde la base (abajo), luego desde el tope (arriba).
        return [("base", p) for p in porcentajes] + [("tope", p) for p in porcentajes]

    # Vertical: primero de izquierda a derecha (tope), luego de derecha a izquierda (base).
    return [("tope", p) for p in porcentajes] + [("base", p) for p in porcentajes]


def calcular_tamanio_fraccion(total: int, porcentaje: float) -> int:
    return max(1, min(total, int(round(total * (porcentaje / 100)))))


def obtener_corte_fuente_fraccion(
        alto: int,
        ancho: int,
        orientacion_corte: OrientacionCorte,
        mitad: str,
        porcentaje: float,
) -> tuple[slice, slice]:
    if orientacion_corte == OrientacionCorte.VERTICAL:
        altura_fraccion = calcular_tamanio_fraccion(alto, porcentaje)
        if mitad == "base":
            return (slice(alto - altura_fraccion, alto), slice(0, ancho))
        return (slice(0, altura_fraccion), slice(0, ancho))

    ancho_fraccion = calcular_tamanio_fraccion(ancho, porcentaje)
    if mitad == "base":
        return (slice(0, alto), slice(ancho - ancho_fraccion, ancho))
    return (slice(0, alto), slice(0, ancho_fraccion))


def cortar_imagen_en_fraccion(
        img,
        orientacion_corte: OrientacionCorte,
        mitad: str,
        porcentaje: float,
):
    alto, ancho = img.shape
    lado_lienzo = max(alto, ancho)

    offset_y = (lado_lienzo - alto) // 2
    offset_x = (lado_lienzo - ancho) // 2

    corte_src = obtener_corte_fuente_fraccion(alto, ancho, orientacion_corte, mitad, porcentaje)

    imagen = crear_lienzo_cuadrado(img, lado_lienzo)
    corte_dst = trasladar_corte_a_lienzo(corte_src, offset_y, offset_x, lado_lienzo)
    imagen[corte_dst] = img[corte_src]

    return imagen


def crear_lienzo_cuadrado(img, lado: int):
    return np.full((lado, lado), 255, dtype=img.dtype)


def trasladar_corte_a_lienzo(
        corte_src: tuple[slice, slice],
        offset_y: int,
        offset_x: int,
        lado_lienzo: int,
) -> tuple[slice, slice]:
    slice_y, slice_x = corte_src

    return (
        trasladar_slice(slice_y, offset_y, lado_lienzo),
        trasladar_slice(slice_x, offset_x, lado_lienzo),
    )


def trasladar_slice(segmento: slice, offset: int, limite: int) -> slice:
    inicio = offset if segmento.start is None else segmento.start + offset
    fin = limite if segmento.stop is None else segmento.stop + offset
    return slice(inicio, fin)

In [7]:
def procesar_perfil(
        ruta_perfil: Path,
        orientacion: OrientacionCorte,
        ruta_destino_clase: Path,
        porcentaje_avance: float,
) -> pd.DataFrame:
    try:
        img = leer_imagen(ruta_perfil)
        nombre_original = obtener_nombre_archivo(ruta_perfil)

        filas: list[dict[str, str | float]] = []

        for mitad, porcentaje in generar_fracciones(orientacion, porcentaje_avance):
            imagen_fraccion = cortar_imagen_en_fraccion(img, orientacion, mitad, porcentaje)
            porcentaje_str = formatear_porcentaje_para_nombre(porcentaje)
            sufijo = f"_{mitad}_{porcentaje_str}"
            ruta_fraccion = crear_ruta_archivo(ruta_destino_clase, nombre_original, sufijo)

            guardar_imagen(ruta=ruta_fraccion, imagen=imagen_fraccion)

            filas.append(
                {
                    "archivo_origen": ruta_perfil.name,
                    "archivo_generado": Path(ruta_fraccion).name,
                    "mitad": mitad,
                    "porcentaje": porcentaje,
                }
            )

        return pd.DataFrame(filas)

    except Exception as e:
        print(f"Error procesando {ruta_perfil}: {e}")
        return pd.DataFrame()


def procesar_clase(
        ruta_origen: Path,
        ruta_destino: Path,
        clase: int | str,
        porcentaje_avance: float,
) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    perfiles = obtener_rutas_perfiles(ruta_origen, clase, extension="png")

    if not perfiles:
        return pd.DataFrame()

    orientacion = determinar_orientacion_corte(clase)

    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_nombre_carpeta_clase(clase)
    )

    for ruta_perfil in perfiles:
        df_perfil = procesar_perfil(ruta_perfil, orientacion, ruta_destino_clase, porcentaje_avance)
        dfs.append(df_perfil)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


def procesar_imagenes(
        ruta_origen: Path,
        ruta_destino: Path,
        clases: list[int | str],
        porcentaje_avance: float,
) -> pd.DataFrame:
    dfs: list[pd.DataFrame] = []

    for clase in clases:
        df_clase = procesar_clase(ruta_origen, ruta_destino, clase, porcentaje_avance)
        dfs.append(df_clase)

    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


In [8]:
def construir_df_desde_mitades(
        df_completos: pd.DataFrame,
        df_mitades: pd.DataFrame,
        how: str = "inner",  # usa "left" para conservar todas las mitades aunque no encuentren match
) -> pd.DataFrame:
    """
    Crea un nuevo dataframe a partir de df_completos y df_mitades:
    - Join por: df_completos.Identificador == stem(df_mitades.archivo_origen)
    - El nuevo 'Identificador' toma el valor de df_mitades.archivo_generado
    - Conserva el resto de columnas de df_completos
    - Agrega columnas: 'archivo_origen' y 'mitad' desde df_mitades
    """
    columnas_mitades = ["archivo_origen", "archivo_generado", "mitad"]
    if "porcentaje" in df_mitades.columns:
        columnas_mitades.append("porcentaje")

    # Nos quedamos solo con las columnas necesarias del dataframe de mitades
    mitades = df_mitades[columnas_mitades].copy()
    mitades["archivo_origen_stem"] = mitades["archivo_origen"].map(lambda nombre: Path(nombre).stem)

    # Join entre ambos dataframes usando el nombre base del archivo original (sin extensión)
    df_join = mitades.merge(
        df_completos,
        left_on="archivo_origen_stem",
        right_on="Identificador",
        how=how,
        validate="many_to_one",  # muchas mitades pueden venir de un mismo original
    )

    # El nuevo Identificador pasa a ser el stem del archivo generado (sin extensión)
    df_join["Identificador"] = df_join["archivo_generado"].map(lambda nombre: Path(nombre).stem)

    # Estructura final: mismas columnas que df_completos + archivo_origen + mitad (+ porcentaje opcional)
    columnas_finales = list(df_completos.columns) + ["archivo_origen", "mitad"]
    if "porcentaje" in df_join.columns:
        columnas_finales.append("porcentaje")

    return df_join[columnas_finales]

In [9]:
from _bootstrap import setup

clases_a_clasificar = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
porcentaje_avance = 10.0
PATHS = setup()

csv_origen: Path = PATHS.DATA_CLASIFICADOS / "metadata.csv"
csv_destino: Path = PATHS.DATA_FRACCIONES / "metadata.csv"

df_completos = pd.read_csv(csv_origen)

df_mitades = procesar_imagenes(
    PATHS.DATA_CLASIFICADOS,
    PATHS.DATA_FRACCIONES,
    clases_a_clasificar,
    porcentaje_avance=porcentaje_avance,
)

df_final = construir_df_desde_mitades(df_completos, df_mitades)
df_final.to_csv(csv_destino, index=False)
df_final.head()

,Identificador,Tiene asa,Tendencia Formal,Borde,Cuello,Cuerpo,Base,Clase Ana,Diametro Borde,Diametro Base,Altura Total,Cronologia,Ubicacion,archivo_origen,mitad,porcentaje
0,mb_03_tope_10,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope,10.0
1,mb_03_tope_20,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope,20.0
2,mb_03_tope_30,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope,30.0
3,mb_03_tope_40,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope,40.0
4,mb_03_tope_50,No,IV,12,B,7,4,1.0,188,47,464,ibero,Marroquies Bajos,mb_03.png,tope,50.0
